
# DACON 딥보이스 탐지 v2 — Domain-Robust Fake Detector + Presence-Gated Top-K

이 노트북은 기존 `RawBoost_SONICS_DynamicMix_25000.ipynb`의 데이터 수집/라이선스/동적 믹싱 구조를 유지하면서,
리더보드에서 **CPS는 높고 ADS가 낮았던 현상**을 줄이기 위해 아래를 변경합니다.

## v2 핵심 변경

1. **Presence와 Fake detection을 분리**
   - Presence: 가벼운 Log-Mel CNN 2-head
   - Fake: 공식 AASIST 3-head + XLS-R 300M dual-graph 3-head
2. **Fake loss를 ADS 비중에 맞춤**
   - File 0.50 / Voice 0.20 / Music 0.30
   - masked BCE + pairwise ranking loss
3. **RawBoost 과적용 완화**
   - AASIST fake: RawBoost 0.15
   - XLS-R fake: RawBoost 0.10
   - clean sample을 충분히 유지
4. **데이터 출처 shortcut 완화**
   - 클래스와 무관하게 source-level mild equalization augmentation 적용
5. **Domain-shift stress validation 추가**
   - train에서 쓰지 않는 band-limit / resample 조합으로 별도 stress score 측정
   - best checkpoint는 normal + stress 성능으로 선택
6. **AASIST + XLS-R fake ensemble**
   - validation normal/stress에서 ensemble weight를 자동 탐색
7. **제출 추론의 max-heavy pooling 제거**
   - Presence-gated Top-K mean
   - Voice fake는 voice-presence가 높은 segment에서만 집계
   - Music fake는 music-presence가 높은 segment에서만 집계
8. **최대 1분 파일 대응**
   - 파일별 최대 8개 4.04초 segment를 uniform sampling하여 L4 60분 제한을 고려

> 주의: Real Music=FMA, Fake Music=SONICS라는 데이터 출처 차이는 완전히 제거할 수 없습니다.
> 이 v2는 그 shortcut을 augmentation과 stress validation으로 줄이는 버전이며,
> 더 높은 일반화를 위해서는 동일/유사 도메인에서 real/fake가 함께 구성된 음악 데이터 추가가 여전히 권장됩니다.



## 라벨과 평가 기준

모델의 최종 출력은 DACON 제출 순서를 유지합니다.

`FILE_FAKE, VOICE_FAKE, MUSIC_FAKE, VOICE_PRESENT, MUSIC_PRESENT`

- FAKE = 1, REAL = 0
- 파일은 음성 또는 음악 중 하나라도 FAKE이면 FILE_FAKE=1
- Voice/Music fake loss는 해당 성분이 존재할 때만 계산
- Presence 모델과 Fake 모델을 분리해서 ADS와 CPS의 역할 충돌을 줄입니다.
- 최종 제출 확률은 hard threshold를 적용하지 않습니다.


## 0. Colab 설치

GPU 런타임(T4/L4/A100)을 선택합니다. SONICS는 생성 없이 Hugging Face에서 두 개의 공식 ZIP을 내려받습니다. 학습 checkpoint와 manifest는 Drive에 저장되어 세션 재시작 후 이어집니다.


In [ ]:
import subprocess
import sys

packages = [
    "kaggle>=1.7", "transformers>=4.57,<5", "accelerate>=1.9", "huggingface_hub>=0.34",
    "librosa==0.10.2.post1", "soundfile>=0.12", "scikit-learn>=1.4",
    "panns-inference==0.1.1", "seaborn>=0.13", "pandas>=2.0",
    "scipy>=1.11", "einops>=0.8", "tqdm>=4.66",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("설치 완료. import 오류가 남으면 런타임을 한 번 재시작하세요.")


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path
from types import SimpleNamespace

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")
warnings.filterwarnings("ignore", category=FutureWarning)

CFG = SimpleNamespace(
    seed=42,
    sample_rate=16_000,
    # 공식 AASIST 입력 길이(64,600 samples @ 16 kHz)에 맞춘다.
    clip_seconds=4.0375,
    clip_samples=64_600,
    source_per_pool=6_250,
    source_split_counts={"train": 5_000, "validation": 625, "audit": 625},
    recipe_counts={"train": 20_000, "validation": 2_500, "audit": 2_500},
    panns_seconds=10,
    panns_batch=8,
    num_workers=0,
)
assert sum(CFG.source_split_counts.values()) == CFG.source_per_pool
assert sum(CFG.recipe_counts.values()) == 25_000

DACON_PROBABILITY_COLUMNS = [
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
]
DACON_TRUTH_COLUMNS = [
    "FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT",
]
HEAD_WEIGHTS = torch.tensor([0.45, 0.18, 0.27, 0.05, 0.05], dtype=torch.float32)
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".amr"}

DRIVE_ROOT = Path("/content/drive/MyDrive/deepvoice_dynamic25k_v2")
LOCAL_ROOT = Path("/content/deepvoice_dynamic25k")
VOICE_ROOT = LOCAL_ROOT / "voice_kaggle"
FMA_ROOT = LOCAL_ROOT / "fma"
SONICS_ROOT = LOCAL_ROOT / "sonics"
REPO_ROOT = LOCAL_ROOT / "repos"
RUN_ROOT = DRIVE_ROOT / "runs"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
for directory in (DRIVE_ROOT, LOCAL_ROOT, VOICE_ROOT, FMA_ROOT, SONICS_ROOT, REPO_ROOT, RUN_ROOT, MANIFEST_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.__version__)
print("source references:", f"{4 * CFG.source_per_pool:,}")
print("dynamic recipes:", f"{sum(CFG.recipe_counts.values()):,}")
print("free disk GB:", round(shutil.disk_usage("/content").free / 1024**3, 1))


## 1. Real/Fake Voice와 FMA 다운로드

Kaggle 데이터셋은 `real/` 9,066개, `fake/` 6,722개 구조이며 약 4.7GB입니다. FMA medium은 small+medium 30초 MP3 25,000개(약 22GiB)와 metadata를 공식 배포 URL에서 받습니다. FMA small만으로는 NoDerivatives를 올바르게 제외한 뒤 6,250곡이 남지 않으므로 medium 묶음을 사용합니다. 압축 파일 hash를 확인하고 재실행 시 기존 파일을 재사용합니다.


In [ ]:
KAGGLE_DATASET = "jayjoshi37/deepfake-audio-dataset-fake-vs-real-speech"
DOWNLOAD_VOICE = True
DOWNLOAD_FMA = True


def configure_kaggle_auth():
    from google.colab import files, userdata
    token = username = key = None
    try:
        token = userdata.get("KAGGLE_API_KEY")
    except Exception:
        pass
    if not token:
        try:
            username = userdata.get("KAGGLE_USERNAME")
            key = userdata.get("KAGGLE_KEY")
        except Exception:
            pass
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    elif username and key:
        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key
    else:
        print("Kaggle Settings에서 받은 kaggle.json을 업로드하세요.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("kaggle.json이 업로드되지 않았습니다.")
        credential_dir = Path("/root/.kaggle")
        credential_dir.mkdir(parents=True, exist_ok=True)
        credential_path = credential_dir / "kaggle.json"
        credential_path.write_bytes(uploaded["kaggle.json"])
        credential_path.chmod(0o600)


def sha1_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha1()
    with Path(path).open("rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


voice_audio = [p for p in VOICE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
if DOWNLOAD_VOICE and not voice_audio:
    configure_kaggle_auth()
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(VOICE_ROOT), "--unzip", "--quiet"],
        check=True,
    )

FMA_FILES = {
    "fma_medium.zip": (
        "https://os.unil.cloud.switch.ch/fma/fma_medium.zip",
        "c67b69ea232021025fca9231fc1c7c1a063ab50b",
    ),
    "fma_metadata.zip": (
        "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip",
        "f0df49ffe5f2a6008d7dc83c6915b31835dfe733",
    ),
}
if DOWNLOAD_FMA:
    for filename, (url, expected_sha1) in FMA_FILES.items():
        archive_path = FMA_ROOT / filename
        if not archive_path.exists():
            subprocess.run(["wget", "-q", "--show-progress", "-O", str(archive_path), url], check=True)
        actual_sha1 = sha1_file(archive_path)
        if actual_sha1 != expected_sha1:
            raise RuntimeError(f"FMA hash mismatch: {filename} {actual_sha1}")
        marker = FMA_ROOT / (filename + ".extracted")
        if not marker.exists():
            subprocess.run(["unzip", "-q", "-o", str(archive_path), "-d", str(FMA_ROOT)], check=True)
            marker.write_text(actual_sha1, encoding="utf-8")

voice_audio = [p for p in VOICE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
fma_audio = [p for p in (FMA_ROOT / "fma_medium").rglob("*.mp3") if p.is_file()]
print("voice audio:", f"{len(voice_audio):,}")
print("FMA medium audio:", f"{len(fma_audio):,}")
if len(voice_audio) < 2 * CFG.source_per_pool:
    raise RuntimeError("Kaggle real/fake 음성 파일이 충분하지 않습니다.")
if len(fma_audio) < CFG.source_per_pool:
    raise RuntimeError("FMA real music 파일이 충분하지 않습니다.")


## 2. SONICS fake music 다운로드

[SONICS 공식 데이터셋](https://huggingface.co/datasets/awsaf49/sonics)은 49,074개의 Suno/Udio 가짜 곡을 10개 ZIP으로 제공합니다. 각 ZIP에는 5,000곡이 들어 있으므로 `part_01.zip`과 `part_02.zip`만 내려받아 10,000곡 중 6,250곡을 선택합니다. 전체 32.6GB 저장소를 받을 필요가 없습니다.

SONICS는 가짜 곡 오디오만 제공하므로 real music은 FMA를 계속 사용합니다. 데이터셋 라이선스는 CC BY-NC 4.0이며, metadata의 `source`, `algorithm`, `label`, `split`, `no_vocal`을 source manifest에 보존합니다.


In [ ]:
from huggingface_hub import hf_hub_download

SONICS_REPO = "awsaf49/sonics"
SONICS_REVISION = "3788dca9f9f11ad92e9097ef4b58eee247661e7f"
SONICS_PARTS = ["fake_songs/part_01.zip", "fake_songs/part_02.zip"]
DOWNLOAD_SONICS = True
DELETE_SONICS_ARCHIVES_AFTER_EXTRACT = True
SONICS_METADATA_PATH = SONICS_ROOT / "fake_songs.csv"
SONICS_AUDIO_ROOT = SONICS_ROOT / "fake_songs"
SONICS_ROOT.mkdir(parents=True, exist_ok=True)


def download_sonics_file(filename):
    return Path(hf_hub_download(
        repo_id=SONICS_REPO,
        repo_type="dataset",
        filename=filename,
        revision=SONICS_REVISION,
        local_dir=str(SONICS_ROOT),
    ))


if DOWNLOAD_SONICS:
    if not SONICS_METADATA_PATH.exists():
        downloaded_metadata = download_sonics_file("fake_songs.csv")
        if downloaded_metadata.resolve() != SONICS_METADATA_PATH.resolve():
            shutil.copy2(downloaded_metadata, SONICS_METADATA_PATH)

    for part_name in SONICS_PARTS:
        part_stem = Path(part_name).stem
        marker = SONICS_ROOT / f".{part_stem}.extracted"
        if marker.exists():
            print("reuse extracted:", part_name)
            continue
        archive_path = download_sonics_file(part_name)
        with zipfile.ZipFile(archive_path) as archive:
            members = [info for info in archive.infolist() if not info.is_dir()]
            if len(members) != 5_000:
                raise RuntimeError(f"SONICS {part_name}: expected 5,000 files, got {len(members):,}")
            archive.extractall(SONICS_ROOT)
        marker.write_text(f"{SONICS_REVISION}\n{len(members)} files\n", encoding="utf-8")
        if DELETE_SONICS_ARCHIVES_AFTER_EXTRACT and archive_path.exists():
            archive_path.unlink()
elif not SONICS_METADATA_PATH.exists() or not SONICS_AUDIO_ROOT.exists():
    raise FileNotFoundError("DOWNLOAD_SONICS=False이면 SONICS metadata와 fake_songs 폴더를 직접 준비해야 합니다.")

sonics_audio = sorted(
    path for path in SONICS_AUDIO_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES and path.stat().st_size > 0
)
sonics_metadata = pd.read_csv(
    SONICS_METADATA_PATH,
    usecols=["filename", "algorithm", "source", "label", "target", "skip_time", "no_vocal", "split"],
    low_memory=False,
)
sonics_metadata = sonics_metadata[sonics_metadata["target"].eq(1)].copy()
sonics_metadata["file_stem"] = sonics_metadata["filename"].astype(str).map(lambda value: Path(value).stem)
if sonics_metadata["no_vocal"].dtype != bool:
    sonics_metadata["no_vocal"] = (
        sonics_metadata["no_vocal"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
    )
sonics_metadata = sonics_metadata.drop_duplicates("file_stem", keep="last")
sonics_metadata_lookup = sonics_metadata.set_index("file_stem").to_dict("index")
sonics_candidates = [
    str(path.resolve()) for path in sonics_audio if path.stem in sonics_metadata_lookup
]
print({
    "SONICS extracted audio": len(sonics_audio),
    "metadata fake rows": len(sonics_metadata),
    "matched candidates": len(sonics_candidates),
    "sources": sonics_metadata.loc[
        sonics_metadata["file_stem"].isin({Path(path).stem for path in sonics_candidates}), "source"
    ].value_counts().to_dict(),
})
if len(sonics_candidates) < CFG.source_per_pool:
    raise RuntimeError(f"SONICS fake song이 {CFG.source_per_pool:,}개 필요합니다.")


## 3. 네 소스 풀에서 각각 정확히 6,250개 선택

Kaggle 폴더의 `real/`, `fake/`를 음성 라벨로 사용합니다. FMA는 metadata에서 small+medium subset을 확인하고 `NoDerivatives` 계열을 제외합니다. SONICS는 다운로드한 10,000곡 중 metadata와 일치하는 가짜 곡을 선택합니다. 모든 선택은 stable hash 정렬을 사용해 재실행해도 동일합니다.


In [ ]:
def stable_int(text):
    return int(hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16], 16)


def select_exact(paths, count, namespace):
    unique_paths = sorted({str(Path(path).resolve()) for path in paths})
    ordered = sorted(unique_paths, key=lambda value: stable_int(f"{CFG.seed}|{namespace}|{value}"))
    if len(ordered) < count:
        raise RuntimeError(f"{namespace}: {count:,}개 필요, {len(ordered):,}개 발견")
    return ordered[:count]


real_voice_all, fake_voice_all, unresolved_voice = [], [], []
for path in voice_audio:
    parts = {part.lower() for part in path.parts}
    if "real" in parts and "fake" not in parts:
        real_voice_all.append(path)
    elif "fake" in parts and "real" not in parts:
        fake_voice_all.append(path)
    else:
        unresolved_voice.append(path)
print({"real_voice": len(real_voice_all), "fake_voice": len(fake_voice_all), "unresolved": len(unresolved_voice)})

tracks_path = FMA_ROOT / "fma_metadata" / "tracks.csv"
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1], low_memory=False)
small_medium_tracks = tracks[
    tracks[("set", "subset")].astype(str).isin({"small", "medium"})
].copy()

def fma_track_path(track_id):
    track_id = int(track_id)
    return FMA_ROOT / "fma_medium" / f"{track_id:06d}"[:3] / f"{track_id:06d}.mp3"

def fma_license_allowed(value):
    value = str(value).lower()
    if not value or value == "nan":
        return False
    compact = re.sub(r"[^a-z0-9]+", "", value)
    normalized = re.sub(r"[^a-z0-9]+", " ", value)
    no_derivatives = (
        "noderivative" in compact
        or "noderivs" in compact
        or "musicsharing" in compact
        or re.search(r"\bby\s+(?:nc\s+)?nd\b", normalized) is not None
    )
    return not no_derivatives

fma_candidates = []
fma_license_lookup = {}
for track_id, row in small_medium_tracks.iterrows():
    path = fma_track_path(track_id)
    license_value = row.get(("track", "license"), "")
    if path.is_file() and path.stat().st_size > 0 and fma_license_allowed(license_value):
        fma_candidates.append(path)
        fma_license_lookup[str(path.resolve())] = str(license_value)

selected = {
    "real_voice": select_exact(real_voice_all, CFG.source_per_pool, "real_voice"),
    "fake_voice": select_exact(fake_voice_all, CFG.source_per_pool, "fake_voice"),
    "real_music": select_exact(fma_candidates, CFG.source_per_pool, "real_music"),
    "fake_music": select_exact(sonics_candidates, CFG.source_per_pool, "fake_music"),
}
assert sum(map(len, selected.values())) == 25_000
if set(selected["real_voice"]) & set(selected["fake_voice"]):
    raise RuntimeError("real/fake voice pool overlap")
display(pd.DataFrame({name: [len(paths)] for name, paths in selected.items()}))
print("FMA eligible after license filter:", f"{len(fma_candidates):,}")


## 4. FMA 보컬 스크리닝

PANNs Cnn14의 AudioSet `Speech`, `Singing`, `Choir`, `Vocal music` 계열 점수로 FMA track의 보컬 가능성을 기록합니다. 모델이 music-only로 잘못 학습하지 않도록, 검출된 track은 real music과 real voice가 함께 존재하는 source로 취급합니다.

스크리닝 결과는 Drive CSV에 누적 저장되어 중단 후 이어집니다. `RUN_FMA_VOCAL_SCREEN=False`는 빠른 디버그에만 사용하세요.


In [ ]:
RUN_FMA_VOCAL_SCREEN = True
FMA_VOCAL_THRESHOLD = 0.20
FMA_SCREEN_PATH = MANIFEST_ROOT / "fma_panns_vocal_screen.csv"

if FMA_SCREEN_PATH.exists():
    fma_screen = pd.read_csv(FMA_SCREEN_PATH)
else:
    fma_screen = pd.DataFrame(columns=["path", "vocal_score", "music_score", "contains_voice", "screen_ok"])

def as_boolean(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

if RUN_FMA_VOCAL_SCREEN:
    from panns_inference import AudioTagging, labels as panns_labels

    vocal_names = [
        "Speech", "Male speech, man speaking", "Female speech, woman speaking",
        "Conversation", "Narration, monologue", "Singing", "Choir", "Vocal music",
    ]
    vocal_indices = [panns_labels.index(name) for name in vocal_names if name in panns_labels]
    music_index = panns_labels.index("Music")
    completed = set(fma_screen.loc[as_boolean(fma_screen["screen_ok"]), "path"].astype(str)) if len(fma_screen) else set()
    missing = [path for path in selected["real_music"] if path not in completed]

    def load_for_panns(path):
        audio, _ = librosa.load(path, sr=32_000, mono=True, duration=CFG.panns_seconds)
        target = 32_000 * CFG.panns_seconds
        if len(audio) < target:
            audio = np.pad(audio, (0, target - len(audio)))
        return np.asarray(audio[:target], dtype=np.float32)

    new_rows = []
    if missing:
        checkpoint_path = DRIVE_ROOT / "panns" / "Cnn14_mAP=0.431.pth"
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        tagger = AudioTagging(checkpoint_path=str(checkpoint_path), device=str(DEVICE))
    for offset in tqdm(range(0, len(missing), CFG.panns_batch), desc="FMA PANNs vocal screen"):
        batch_paths = missing[offset:offset + CFG.panns_batch]
        batch_audio, ok_flags = [], []
        for path in batch_paths:
            try:
                batch_audio.append(load_for_panns(path))
                ok_flags.append(True)
            except Exception:
                batch_audio.append(np.zeros(32_000 * CFG.panns_seconds, dtype=np.float32))
                ok_flags.append(False)
        clipwise, _ = tagger.inference(np.stack(batch_audio))
        for path, scores, ok in zip(batch_paths, clipwise, ok_flags):
            vocal_score = float(np.max(scores[vocal_indices])) if vocal_indices else 0.0
            new_rows.append({
                "path": path, "vocal_score": vocal_score,
                "music_score": float(scores[music_index]),
                "contains_voice": bool(vocal_score >= FMA_VOCAL_THRESHOLD),
                "screen_ok": bool(ok),
            })
        if len(new_rows) >= 128 or offset + CFG.panns_batch >= len(missing):
            fma_screen = pd.concat([fma_screen, pd.DataFrame(new_rows)], ignore_index=True)
            fma_screen = fma_screen.drop_duplicates("path", keep="last")
            fma_screen.to_csv(FMA_SCREEN_PATH, index=False, encoding="utf-8")
            new_rows = []
    if missing:
        del tagger
    gc.collect()
    torch.cuda.empty_cache()
else:
    fma_screen = pd.DataFrame({
        "path": selected["real_music"], "vocal_score": 0.0, "music_score": np.nan,
        "contains_voice": False, "screen_ok": False,
    })
    print("경고: FMA 보컬 스크리닝을 끈 상태입니다. music-only 라벨 노이즈가 생길 수 있습니다.")

fma_screen = fma_screen[fma_screen["path"].isin(selected["real_music"])].copy()
fma_screen["contains_voice"] = as_boolean(fma_screen["contains_voice"])
fma_screen["screen_ok"] = as_boolean(fma_screen["screen_ok"])
print("FMA screened:", len(fma_screen), "vocal-like:", int(fma_screen["contains_voice"].astype(bool).sum()))
display(fma_screen.describe(include="all").T)


## 5. Source manifest와 누수 없는 split

각 풀의 6,250개를 5,000/625/625로 정확히 나눕니다. 동일 source path가 split을 넘지 않는지 검사하고, 데이터 출처·license·FMA 보컬 스크리닝 값을 manifest에 보존합니다.


In [ ]:
fma_voice_lookup = dict(zip(fma_screen["path"].astype(str), fma_screen["contains_voice"].astype(bool)))
rows = []
for pool, paths in selected.items():
    ordered = sorted(paths, key=lambda value: stable_int(f"split|{CFG.seed}|{pool}|{value}"))
    boundaries = {}
    start = 0
    for split_name, count in CFG.source_split_counts.items():
        boundaries[split_name] = ordered[start:start + count]
        start += count
    for split_name, split_paths in boundaries.items():
        for path in split_paths:
            sonics_record = sonics_metadata_lookup.get(Path(path).stem, {}) if pool == "fake_music" else {}
            if pool == "real_music":
                contains_voice = bool(fma_voice_lookup.get(path, False))
            elif pool == "fake_music":
                contains_voice = not bool(sonics_record.get("no_vocal", False))
            else:
                contains_voice = pool.endswith("voice")
            rows.append({
                "source_id": hashlib.sha256(f"{pool}|{path}".encode()).hexdigest()[:20],
                "pool": pool, "split": split_name, "path": path,
                "contains_voice": contains_voice,
                "voice_fake": int(pool == "fake_voice" or (pool == "fake_music" and contains_voice)),
                "contains_music": pool.endswith("music"),
                "music_fake": int(pool == "fake_music"),
                "license": (
                    "CC-BY-SA-4.0" if pool.endswith("voice") else
                    fma_license_lookup.get(path, "FMA artist-selected") if pool == "real_music" else
                    "CC-BY-NC-4.0"
                ),
                "origin": (
                    KAGGLE_DATASET if pool.endswith("voice") else
                    "FMA medium" if pool == "real_music" else SONICS_REPO
                ),
                "source_detail": str(sonics_record.get("source", "")),
                "generation_algorithm": str(sonics_record.get("algorithm", "")),
                "sonics_label": str(sonics_record.get("label", "")),
                "sonics_original_split": str(sonics_record.get("split", "")),
            })
source_manifest = pd.DataFrame(rows)
assert len(source_manifest) == 25_000
assert source_manifest["source_id"].is_unique
assert source_manifest.groupby("pool").size().eq(CFG.source_per_pool).all()
assert source_manifest.groupby(["pool", "split"]).size().to_dict() == {
    (pool, split_name): count
    for pool in selected for split_name, count in CFG.source_split_counts.items()
}
if source_manifest.groupby("path")["split"].nunique().max() != 1:
    raise RuntimeError("source path가 여러 split에 존재합니다.")
SOURCE_MANIFEST_PATH = MANIFEST_ROOT / "source_manifest_25000.csv"
source_manifest.to_csv(SOURCE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(source_manifest["pool"], source_manifest["split"], margins=True))
display(source_manifest.groupby(["pool", "license"]).size().rename("count").reset_index())
display(source_manifest[source_manifest["pool"].eq("fake_music")][
    ["source_detail", "generation_algorithm", "sonics_label", "contains_voice"]
].value_counts().rename("count").reset_index().head(30))
print("saved:", SOURCE_MANIFEST_PATH)


## 6. 정확히 25,000개 Dynamic Mix recipe

여덟 조합을 균형 배치합니다: RV, FV, RM, FM, RV+RM, FV+RM, RV+FM, FV+FM. Recipe CSV에는 실제 waveform 대신 split, 조합, seed를 저장합니다. Train은 `epoch`을 seed에 포함해 매 epoch 다른 source/crop/SNR/layout을 만들고 validation/audit은 고정합니다.


In [ ]:
RECIPE_TYPES = ["rv", "fv", "rm", "fm", "rv_rm", "fv_rm", "rv_fm", "fv_fm"]


def build_recipe_manifest():
    recipe_rows = []
    for split_name, count in CFG.recipe_counts.items():
        recipe_types = [RECIPE_TYPES[index % len(RECIPE_TYPES)] for index in range(count)]
        random.Random(CFG.seed + stable_int(split_name)).shuffle(recipe_types)
        for index, recipe_type in enumerate(recipe_types):
            recipe_rows.append({
                "recipe_id": f"{split_name}_{index:05d}",
                "split": split_name, "recipe_type": recipe_type,
                "seed": stable_int(f"recipe|{CFG.seed}|{split_name}|{index}|{recipe_type}") % (2**31 - 1),
            })
    return pd.DataFrame(recipe_rows)


recipe_manifest = build_recipe_manifest()
assert len(recipe_manifest) == 25_000
assert recipe_manifest["recipe_id"].is_unique
assert recipe_manifest.groupby("split").size().to_dict() == CFG.recipe_counts
RECIPE_MANIFEST_PATH = MANIFEST_ROOT / "dynamic_mix_recipes_25000.csv"
recipe_manifest.to_csv(RECIPE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(recipe_manifest["recipe_type"], recipe_manifest["split"], margins=True))
print("saved:", RECIPE_MANIFEST_PATH)


## 7. 공식 AASIST·RawBoost 소스와 오디오 로더

SoundFile → librosa → FFmpeg 순서로 WAV/MP3/FLAC 등을 처리합니다. 긴 파일은 random crop, 짧은 파일은 반복 후 crop합니다. RawBoost는 train 최종 mixture에만 적용합니다.


In [ ]:
repositories = {
    "aasist": "https://github.com/clovaai/aasist.git",
    "rawboost": "https://github.com/TakHemlata/RawBoost-antispoofing.git",
}
repo_commits = {}
for name, url in repositories.items():
    destination = REPO_ROOT / name
    if not destination.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)
    repo_commits[name] = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True,
    ).strip()
print(repo_commits)

rawboost_path = REPO_ROOT / "rawboost" / "RawBoost.py"
rawboost_spec = importlib.util.spec_from_file_location("official_rawboost", rawboost_path)
RAWBOOST = importlib.util.module_from_spec(rawboost_spec)
assert rawboost_spec.loader is not None
rawboost_spec.loader.exec_module(RAWBOOST)


def decode_audio(path):
    path = str(path)
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        waveform = torch.from_numpy(audio.mean(axis=1))
    except Exception:
        try:
            audio, sample_rate = librosa.load(path, sr=None, mono=True)
            waveform = torch.from_numpy(np.asarray(audio, dtype=np.float32))
        except Exception:
            decoded = subprocess.run(
                [
                    "ffmpeg", "-hide_banner", "-loglevel", "error", "-i", path,
                    "-ac", "1", "-ar", str(CFG.sample_rate), "-f", "f32le", "pipe:1",
                ],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=90,
            )
            waveform = torch.from_numpy(np.frombuffer(decoded.stdout, dtype="<f4").copy())
            sample_rate = CFG.sample_rate
    if waveform.numel() == 0:
        raise ValueError(f"empty audio: {path}")
    if sample_rate != CFG.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, CFG.sample_rate)
    waveform = waveform.float().nan_to_num().clamp(-1, 1)
    if waveform.numel() == 0:
        raise ValueError(f"empty after resample: {path}")
    return waveform


def crop_or_repeat(waveform, rng, training):
    if waveform.numel() < CFG.clip_samples:
        waveform = waveform.repeat(math.ceil(CFG.clip_samples / waveform.numel()))
    maximum_start = waveform.numel() - CFG.clip_samples
    start = rng.randint(0, maximum_start) if training and maximum_start else maximum_start // 2
    return waveform[start:start + CFG.clip_samples].clone()


## 8. DynamicMixDataset

두 성분의 RMS를 맞춘 뒤 voice-to-music SNR을 -12~+12dB에서 샘플링합니다. overlap/partial/sequential layout을 섞고, 최종 peak도 0.65~0.98 사이로 바꿔 “크면 mixed”라는 지름길을 막습니다.


In [ ]:

RECIPE_COMPONENTS = {
    "rv": ["real_voice"], "fv": ["fake_voice"],
    "rm": ["real_music"], "fm": ["fake_music"],
    "rv_rm": ["real_voice", "real_music"],
    "fv_rm": ["fake_voice", "real_music"],
    "rv_fm": ["real_voice", "fake_music"],
    "fv_fm": ["fake_voice", "fake_music"],
}


def rms_normalize(waveform, target_db):
    rms = waveform.square().mean().clamp_min(1e-8).sqrt()
    target = 10 ** (target_db / 20)
    return waveform * (target / rms)


def fit_clip_length(waveform):
    waveform = waveform.float().nan_to_num()
    if waveform.numel() == 0:
        return torch.zeros(CFG.clip_samples)
    if waveform.numel() < CFG.clip_samples:
        waveform = waveform.repeat(math.ceil(CFG.clip_samples / waveform.numel()))
    return waveform[:CFG.clip_samples]


def apply_temporal_layout(waveforms, rng):
    if len(waveforms) < 2:
        return waveforms, "single"
    choice = rng.random()
    if choice < 0.65:
        return waveforms, "overlap"
    length = CFG.clip_samples
    if choice < 0.85:
        result = []
        for waveform in waveforms:
            active = rng.randint(CFG.sample_rate, length)
            start = rng.randint(0, length - active)
            mask = torch.zeros(length)
            mask[start:start + active] = 1
            result.append(waveform * mask)
        return result, "partial"
    boundary = rng.randint(int(0.35 * length), int(0.65 * length))
    first_mask = torch.zeros(length); first_mask[:boundary] = 1
    second_mask = 1 - first_mask
    return [waveforms[0] * first_mask, waveforms[1] * second_mask], "sequential"


def source_equalize_augment(waveform, rng):
    """Class-independent mild transform to reduce FMA/SONICS/Kaggle source shortcuts."""
    mode = rng.choice(["resample12k", "mulaw", "lowpass", "gain"])
    if mode == "resample12k":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 12_000)
        waveform = torchaudio.functional.resample(waveform, 12_000, CFG.sample_rate)
    elif mode == "mulaw":
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
    elif mode == "lowpass":
        cutoff = rng.uniform(5200, 7200)
        waveform = torchaudio.functional.lowpass_biquad(waveform, CFG.sample_rate, cutoff)
    else:
        waveform = waveform * 10 ** (rng.uniform(-4, 4) / 20)
    return fit_clip_length(waveform)


def communication_augment(waveform, rng):
    mode = rng.choice(["telephone", "mulaw", "noise", "gain_clip"])
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "mulaw":
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
    elif mode == "noise":
        power = waveform.square().mean().clamp_min(1e-8)
        snr_db = rng.uniform(15, 35)
        generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
        noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
        waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
    else:
        waveform = waveform * 10 ** (rng.uniform(-6, 4) / 20)
        limit = rng.uniform(0.45, 0.95)
        waveform = waveform.clamp(-limit, limit) / limit
    return fit_clip_length(waveform)


def domain_shift_stress_augment(waveform, rng):
    """Held-out validation transform. Do not use this exact policy in training."""
    mode = rng.choice(["bandlimit", "resample11k", "band_noise", "double_resample"])
    if mode == "bandlimit":
        low = rng.uniform(120, 350)
        high = rng.uniform(3800, 6200)
        waveform = torchaudio.functional.highpass_biquad(waveform, CFG.sample_rate, low)
        waveform = torchaudio.functional.lowpass_biquad(waveform, CFG.sample_rate, high)
    elif mode == "resample11k":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 11_025)
        waveform = torchaudio.functional.resample(waveform, 11_025, CFG.sample_rate)
    elif mode == "band_noise":
        waveform = torchaudio.functional.lowpass_biquad(waveform, CFG.sample_rate, rng.uniform(4500, 6500))
        power = waveform.square().mean().clamp_min(1e-8)
        snr_db = rng.uniform(18, 30)
        generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
        noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
        waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
    else:
        mid_sr = rng.choice([9000, 12000, 14000])
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, mid_sr)
        waveform = torchaudio.functional.resample(waveform, mid_sr, CFG.sample_rate)
    return fit_clip_length(waveform)


class DynamicMixDataset(Dataset):
    def __init__(
        self,
        recipes,
        sources,
        training=False,
        rawboost_p=0.0,
        communication_p=0.0,
        source_equalize_p=0.0,
        stress=False,
    ):
        self.recipes = recipes.reset_index(drop=True)
        self.training = training
        self.rawboost_p = float(rawboost_p)
        self.communication_p = float(communication_p)
        self.source_equalize_p = float(source_equalize_p)
        self.stress = bool(stress)
        self.epoch = 0
        self.pools = {
            pool: frame.to_dict("records")
            for pool, frame in sources.groupby("pool", sort=False)
        }
        for pool in RECIPE_COMPONENTS.values():
            for name in pool:
                if not self.pools.get(name):
                    raise RuntimeError(f"empty source pool: {name}")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.recipes)

    def _load_from_pool(self, pool_name, rng):
        errors = []
        for _ in range(5):
            entry = self.pools[pool_name][rng.randrange(len(self.pools[pool_name]))]
            try:
                waveform = crop_or_repeat(decode_audio(entry["path"]), rng, self.training)
                if self.training and rng.random() < self.source_equalize_p:
                    waveform = source_equalize_augment(waveform, rng)
                return waveform, entry
            except Exception as exc:
                errors.append(f"{entry['path']}: {repr(exc)}")
        raise RuntimeError(" | ".join(errors))

    def __getitem__(self, index):
        recipe = self.recipes.iloc[index]
        epoch = self.epoch if self.training else 0
        rng = random.Random(stable_int(f"mix-v2|{recipe.seed}|{epoch}|{index}|stress={self.stress}"))
        components = RECIPE_COMPONENTS[recipe.recipe_type]
        waveforms, entries = [], []
        try:
            for pool_name in components:
                waveform, entry = self._load_from_pool(pool_name, rng)
                waveforms.append(waveform)
                entries.append((pool_name, entry))
        except Exception as exc:
            return {
                "audio": torch.zeros(CFG.clip_samples), "target": torch.zeros(5),
                "mask": torch.ones(5), "id": recipe.recipe_id,
                "recipe_type": recipe.recipe_type, "valid": torch.tensor(False),
                "error": repr(exc), "layout": "error",
            }

        if len(waveforms) == 2:
            snr_db = rng.uniform(-12, 12)
            waveforms[0] = rms_normalize(waveforms[0], -22 + snr_db / 2)
            waveforms[1] = rms_normalize(waveforms[1], -22 - snr_db / 2)
        else:
            waveforms[0] = rms_normalize(waveforms[0], rng.uniform(-27, -17))

        waveforms, layout = apply_temporal_layout(waveforms, rng)
        mixed = torch.stack(waveforms).sum(0)

        target = torch.zeros(5, dtype=torch.float32)
        voice_present = voice_fake = music_present = music_fake = 0
        for pool_name, entry in entries:
            if pool_name.endswith("voice"):
                voice_present = 1
                voice_fake = max(voice_fake, int(pool_name == "fake_voice"))
            if pool_name.endswith("music"):
                music_present = 1
                music_fake = max(music_fake, int(pool_name == "fake_music"))
                if bool(entry.get("contains_voice", False)):
                    voice_present = 1
                    voice_fake = max(voice_fake, int(pool_name == "fake_music"))
        file_fake = max(voice_fake, music_fake)
        target[:] = torch.tensor([file_fake, voice_fake, music_fake, voice_present, music_present])
        mask = torch.tensor([1, voice_present, music_present, 1, 1], dtype=torch.float32)

        # RawBoost is intentionally reduced in v2 to preserve synthetic artifacts.
        if self.training and rng.random() < self.rawboost_p:
            values = mixed.numpy()
            values = RAWBOOST.LnL_convolutive_noise(
                values, N_f=5, nBands=5, minF=20, maxF=8000,
                minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
                minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20,
                fs=CFG.sample_rate,
            )
            values = RAWBOOST.ISD_additive_noise(values, P=10, g_sd=2)
            mixed = torch.from_numpy(np.asarray(RAWBOOST.normWav(values, 0), dtype=np.float32))

        if self.training and rng.random() < self.communication_p:
            mixed = communication_augment(mixed, rng)

        if self.stress and not self.training:
            mixed = domain_shift_stress_augment(mixed, rng)

        mixed = fit_clip_length(mixed)
        peak = mixed.abs().max().clamp_min(1e-8)
        mixed = mixed / peak * rng.uniform(0.65, 0.98)

        if not torch.isfinite(mixed).all():
            return {
                "audio": torch.zeros(CFG.clip_samples), "target": target, "mask": mask,
                "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
                "valid": torch.tensor(False), "error": "NaN/Inf after mixing", "layout": layout,
            }

        return {
            "audio": mixed.float().clamp(-1, 1), "target": target, "mask": mask,
            "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
            "valid": torch.tensor(True), "error": "", "layout": layout,
        }


source_by_split = {
    split_name: source_manifest[source_manifest["split"] == split_name].copy()
    for split_name in CFG.source_split_counts
}
recipe_by_split = {
    split_name: recipe_manifest[recipe_manifest["split"] == split_name].copy()
    for split_name in CFG.recipe_counts
}

preview_dataset = DynamicMixDataset(
    recipe_by_split["validation"].head(8),
    source_by_split["validation"],
)
preview_rows = []
for index in range(len(preview_dataset)):
    item = preview_dataset[index]
    preview_rows.append({
        "id": item["id"], "type": item["recipe_type"],
        **dict(zip(DACON_TRUTH_COLUMNS, item["target"].tolist()))
    })
display(pd.DataFrame(preview_rows))


## 9. DACON 공식 지표와 masked multi-task loss

[DACON 공식 평가 페이지](https://dacon.io/competitions/official/236749/overview/evaluation)의 계산을 그대로 사용합니다.

- `ADS = 0.5 × (1 - File EER) + 0.2 × (1 - Voice EER) + 0.3 × (1 - Music EER)`
- `CPS = 0.5 × Voice Presence ROC-AUC + 0.5 × Music Presence ROC-AUC`
- `Score = 0.9 × ADS + 0.1 × CPS` (높을수록 좋음)
- FAKE가 양성 클래스 `1`이며, Voice/Music EER은 해당 성분이 존재하는 샘플에서만 계산합니다.

Loss weight는 최종 Score의 각 항 가중치를 그대로 펼친 값입니다: File 0.45, Voice Fake 0.18, Music Fake 0.27, Voice Presence 0.05, Music Presence 0.05. 성분이 없는 fake head는 loss mask에서 제외합니다.


In [ ]:
OFFICIAL_METRIC_WEIGHTS = {
    "score_ads": 0.9,
    "score_cps": 0.1,
    "ads_file": 0.5,
    "ads_voice": 0.2,
    "ads_music": 0.3,
    "cps_voice_presence": 0.5,
    "cps_music_presence": 0.5,
}


def _official_binary_inputs(y_true, y_score, metric_name):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.shape != y_score.shape or y_true.size == 0:
        raise ValueError(f"{metric_name}: shape/empty input error")
    if not np.isfinite(y_true).all() or not np.isfinite(y_score).all():
        raise ValueError(f"{metric_name}: NaN/Inf input")
    if np.unique(y_true).size < 2:
        raise ValueError(f"{metric_name}: positive/negative classes are both required")
    return y_true, y_score


def equal_error_rate(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "EER")
    # DACON 평가 페이지에 공개된 EER 구현과 동일합니다.
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1, drop_intermediate=False)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return float(eer)


def official_roc_auc(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "ROC-AUC")
    return float(roc_auc_score(y_true, y_score))


def dacon_official_score(y_true, y_pred):
    file_eer = equal_error_rate(y_true["FILE_FAKE"], y_pred["FILE_FAKE_PROB"])
    voice_mask = y_true["VOICE_PRESENT"].eq(1)
    music_mask = y_true["MUSIC_PRESENT"].eq(1)
    voice_eer = equal_error_rate(y_true.loc[voice_mask, "VOICE_FAKE"], y_pred.loc[voice_mask, "VOICE_FAKE_PROB"])
    music_eer = equal_error_rate(y_true.loc[music_mask, "MUSIC_FAKE"], y_pred.loc[music_mask, "MUSIC_FAKE_PROB"])
    voice_auc = official_roc_auc(y_true["VOICE_PRESENT"], y_pred["VOICE_PRESENT_PROB"])
    music_auc = official_roc_auc(y_true["MUSIC_PRESENT"], y_pred["MUSIC_PRESENT_PROB"])
    ads = (
        OFFICIAL_METRIC_WEIGHTS["ads_file"] * (1 - file_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_voice"] * (1 - voice_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_music"] * (1 - music_eer)
    )
    cps = (
        OFFICIAL_METRIC_WEIGHTS["cps_voice_presence"] * voice_auc
        + OFFICIAL_METRIC_WEIGHTS["cps_music_presence"] * music_auc
    )
    score = (
        OFFICIAL_METRIC_WEIGHTS["score_ads"] * ads
        + OFFICIAL_METRIC_WEIGHTS["score_cps"] * cps
    )
    return {
        "file_eer": file_eer, "voice_eer": voice_eer, "music_eer": music_eer,
        "voice_presence_auc": voice_auc, "music_presence_auc": music_auc,
        "ads": ads, "cps": cps, "score": score,
    }


def masked_multitask_loss(logits, targets, masks):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    weights = HEAD_WEIGHTS.to(logits.device).unsqueeze(0) * masks
    return (loss * weights).sum() / weights.sum().clamp_min(1e-8)


def report_from_arrays(targets, predictions):
    y_true = pd.DataFrame(targets, columns=DACON_TRUTH_COLUMNS)
    y_pred = pd.DataFrame(predictions, columns=DACON_PROBABILITY_COLUMNS)
    return dacon_official_score(y_true, y_pred)



## 10. v2 모델 구조

### Presence branch
가벼운 Log-Mel CNN으로 `VOICE_PRESENT`, `MUSIC_PRESENT` 두 값만 학습합니다.
Presence는 semantic task이므로 복잡한 anti-spoofing backbone을 사용하지 않습니다.

### Fake branches
두 detector를 별도 학습하고 validation에서 ensemble weight를 찾습니다.

1. **AASIST-Fake3**: 공식 AASIST 구조, 출력 3개(File/Voice/Music Fake)
2. **XLSR-Graph-Fake3**: `facebook/wav2vec2-xls-r-300m` + AASIST-inspired temporal/channel dual graph

Fake 모델은 masked BCE와 pairwise ranking loss를 함께 사용합니다.


In [ ]:

from transformers import AutoModel

XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"


class PresenceCNN(nn.Module):
    def __init__(self, dropout=0.20):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=1024,
            win_length=400,
            hop_length=160,
            n_mels=96,
            f_min=20,
            f_max=7600,
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 2))

    def forward(self, audio):
        feature = torch.log(self.mel(audio).clamp_min(1e-6))
        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (feature.std((-2, -1), keepdim=True) + 1e-5)
        return self.head(self.encoder(feature.unsqueeze(1)))


def build_aasist_fake3():
    repo = REPO_ROOT / "aasist"
    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from models.AASIST import Model as AASISTModel

    class Wrapper(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = AASISTModel(model_config)
            self.net.out_layer = nn.Linear(self.net.out_layer.in_features, 3)

        def forward(self, audio):
            _, logits = self.net(audio, Freq_aug=False)
            return logits

    return Wrapper()


class AttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, 4, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attention(z, z, z, need_weights=False)[0]
        return x + self.ff(self.norm2(x))


class XLSRGraphFake(nn.Module):
    """XLS-R + AASIST-inspired temporal/channel dual graph, 3 fake logits."""
    def __init__(self, model_name=XLSR_MODEL, freeze=True, dim=128, dropout=0.20):
        super().__init__()
        self.ssl = AutoModel.from_pretrained(model_name)
        self.hidden = self.ssl.config.hidden_size
        try:
            self.ssl.gradient_checkpointing_enable()
        except Exception:
            pass
        if freeze:
            for parameter in self.ssl.parameters():
                parameter.requires_grad = False

        self.projection = nn.Linear(self.hidden, dim)
        self.feature_projection = nn.Linear(8, dim)
        self.time_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.feature_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.head = nn.Sequential(
            nn.LayerNorm(dim * 4), nn.Dropout(dropout),
            nn.Linear(dim * 4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 3),
        )

    def features(self, audio):
        audio = (audio - audio.mean(1, keepdim=True)) / (audio.std(1, keepdim=True) + 1e-5)
        if any(parameter.requires_grad for parameter in self.ssl.parameters()):
            return self.ssl(audio).last_hidden_state
        with torch.no_grad():
            return self.ssl(audio).last_hidden_state

    def unfreeze_last(self, count=4):
        for parameter in self.ssl.parameters():
            parameter.requires_grad = False
        for layer in self.ssl.encoder.layers[-int(count):]:
            for parameter in layer.parameters():
                parameter.requires_grad = True

    def forward(self, audio):
        hidden = self.projection(self.features(audio))
        time_nodes = F.adaptive_avg_pool1d(hidden.transpose(1, 2), 64).transpose(1, 2)
        feature_nodes = self.feature_projection(F.adaptive_avg_pool1d(hidden.transpose(1, 2), 8))
        time_nodes = self.time_graph(time_nodes)
        feature_nodes = self.feature_graph(feature_nodes)
        pooled = torch.cat(
            [time_nodes.mean(1), time_nodes.amax(1), feature_nodes.mean(1), feature_nodes.amax(1)],
            dim=-1,
        )
        return self.head(pooled)



## 11. Loss와 local metrics

- Presence: 2-head BCE
- Fake: masked BCE + pairwise ranking loss
- Fake best checkpoint: `0.65 × normal ADS + 0.35 × stress ADS`
- Presence best checkpoint: `0.75 × normal CPS + 0.25 × stress CPS`

Ranking loss는 EER/AUC에서 중요한 score ordering을 직접 보조합니다.


In [ ]:

FAKE_HEAD_WEIGHTS = torch.tensor([0.50, 0.20, 0.30], dtype=torch.float32)
RANKING_WEIGHT = 0.20


def presence_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets[:, 3:5])


def pairwise_ranking_loss(logits, targets3, mask3):
    pieces = []
    for head in range(3):
        valid = mask3[:, head] > 0.5
        scores = logits[:, head]
        labels = targets3[:, head]
        pos = scores[valid & (labels > 0.5)]
        neg = scores[valid & (labels <= 0.5)]
        if pos.numel() and neg.numel():
            diff = pos[:, None] - neg[None, :]
            pieces.append(F.softplus(-diff).mean())
    if not pieces:
        return logits.sum() * 0.0
    return torch.stack(pieces).mean()


def fake_loss(logits, targets, masks):
    targets3 = targets[:, :3]
    mask3 = masks[:, :3]
    bce = F.binary_cross_entropy_with_logits(logits, targets3, reduction="none")
    weights = FAKE_HEAD_WEIGHTS.to(logits.device).unsqueeze(0) * mask3
    bce = (bce * weights).sum() / weights.sum().clamp_min(1e-8)
    rank = pairwise_ranking_loss(logits, targets3, mask3)
    return (1 - RANKING_WEIGHT) * bce + RANKING_WEIGHT * rank


def fake_report(targets, fake_predictions):
    y_true = pd.DataFrame(targets, columns=DACON_TRUTH_COLUMNS)
    preds = np.asarray(fake_predictions)
    file_eer = equal_error_rate(y_true["FILE_FAKE"], preds[:, 0])
    voice_mask = y_true["VOICE_PRESENT"].eq(1)
    music_mask = y_true["MUSIC_PRESENT"].eq(1)
    voice_eer = equal_error_rate(y_true.loc[voice_mask, "VOICE_FAKE"], preds[voice_mask.to_numpy(), 1])
    music_eer = equal_error_rate(y_true.loc[music_mask, "MUSIC_FAKE"], preds[music_mask.to_numpy(), 2])
    ads = 0.5 * (1 - file_eer) + 0.2 * (1 - voice_eer) + 0.3 * (1 - music_eer)
    return {
        "file_eer": file_eer,
        "voice_eer": voice_eer,
        "music_eer": music_eer,
        "ads": float(ads),
    }


def presence_report(targets, presence_predictions):
    y_true = np.asarray(targets)
    pred = np.asarray(presence_predictions)
    voice_auc = official_roc_auc(y_true[:, 3], pred[:, 0])
    music_auc = official_roc_auc(y_true[:, 4], pred[:, 1])
    cps = 0.5 * voice_auc + 0.5 * music_auc
    return {
        "voice_presence_auc": voice_auc,
        "music_presence_auc": music_auc,
        "cps": float(cps),
    }


def full_report(targets, fake_predictions, presence_predictions):
    targets = np.asarray(targets)
    fake_predictions = np.asarray(fake_predictions)
    presence_predictions = np.asarray(presence_predictions)
    y_true = pd.DataFrame(targets, columns=DACON_TRUTH_COLUMNS)
    y_pred = pd.DataFrame({
        "FILE_FAKE_PROB": fake_predictions[:, 0],
        "VOICE_FAKE_PROB": fake_predictions[:, 1],
        "MUSIC_FAKE_PROB": fake_predictions[:, 2],
        "VOICE_PRESENT_PROB": presence_predictions[:, 0],
        "MUSIC_PRESENT_PROB": presence_predictions[:, 1],
    })
    return dacon_official_score(y_true, y_pred)



## 12. Train / validation / stress loader

Validation source 파일은 train과 공유하지 않습니다.  
Stress validation은 validation source 중 고정 1,000 recipe에 held-out channel 변형을 적용합니다.


In [ ]:

STRESS_VALIDATION_SAMPLES = 1000


def make_task_loaders(batch, eval_batch, rawboost_p, communication_p, source_equalize_p):
    train_dataset = DynamicMixDataset(
        recipe_by_split["train"],
        source_by_split["train"],
        training=True,
        rawboost_p=rawboost_p,
        communication_p=communication_p,
        source_equalize_p=source_equalize_p,
        stress=False,
    )
    validation_dataset = DynamicMixDataset(
        recipe_by_split["validation"],
        source_by_split["validation"],
        training=False,
        stress=False,
    )
    stress_recipes = recipe_by_split["validation"].sample(
        n=min(STRESS_VALIDATION_SAMPLES, len(recipe_by_split["validation"])),
        random_state=CFG.seed + 991,
    ).reset_index(drop=True)
    stress_dataset = DynamicMixDataset(
        stress_recipes,
        source_by_split["validation"],
        training=False,
        stress=True,
    )

    loader_kwargs = dict(
        num_workers=CFG.num_workers,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=False,
    )
    train_loader = DataLoader(
        train_dataset, batch_size=batch, shuffle=True, drop_last=False, **loader_kwargs
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=eval_batch, shuffle=False, drop_last=False, **loader_kwargs
    )
    stress_loader = DataLoader(
        stress_dataset, batch_size=eval_batch, shuffle=False, drop_last=False, **loader_kwargs
    )
    return train_dataset, train_loader, validation_loader, stress_loader



## 13. 공통 train/eval 함수

Presence와 Fake 모델을 같은 5-head loss로 학습하지 않습니다.


In [ ]:

def _valid_batch(batch):
    valid = batch["valid"].bool()
    if not valid.any():
        return None
    audio = batch["audio"][valid].to(DEVICE, non_blocking=True)
    targets = batch["target"][valid].to(DEVICE, non_blocking=True)
    masks = batch["mask"][valid].to(DEVICE, non_blocking=True)
    return audio, targets, masks


def train_epoch(model, loader, task, optimizer, scaler, grad_accum=1, scheduler=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = total_n = 0
    pbar = tqdm(loader, desc=f"train-{task}", leave=False, dynamic_ncols=True)

    for step, batch in enumerate(pbar, 1):
        valid_data = _valid_batch(batch)
        if valid_data is None:
            continue
        audio, targets, masks = valid_data

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=DEVICE.type == "cuda",
        ):
            logits = model(audio)
            unscaled = presence_loss(logits, targets) if task == "presence" else fake_loss(logits, targets, masks)
            loss = unscaled / grad_accum

        scaler.scale(loss).backward()
        if step % grad_accum == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()

        n = int(audio.shape[0])
        total_loss += float(unscaled.detach().cpu()) * n
        total_n += n
        pbar.set_postfix(loss=f"{total_loss/max(total_n,1):.4f}")

    return total_loss / max(total_n, 1)


@torch.inference_mode()
def predict_task(model, loader, task):
    model.eval()
    targets_all, preds_all = [], []
    for batch in tqdm(loader, desc=f"eval-{task}", leave=False, dynamic_ncols=True):
        valid_data = _valid_batch(batch)
        if valid_data is None:
            continue
        audio, targets, _ = valid_data
        logits = model(audio)
        preds = torch.sigmoid(logits).float().cpu().numpy()
        targets_all.append(targets.float().cpu().numpy())
        preds_all.append(preds)
    if not targets_all:
        raise RuntimeError("No valid samples")
    return np.concatenate(targets_all), np.concatenate(preds_all)



## 14. Presence CNN 학습

Presence는 기존 리더보드에서도 상대적으로 잘 일반화했으므로, 작은 CNN을 사용하고 과도한 RawBoost는 적용하지 않습니다.


In [ ]:

PRESENCE_CFG = dict(
    batch=32,
    eval_batch=64,
    epochs=15,
    lr=3e-4,
    weight_decay=1e-4,
    patience=5,
    rawboost_p=0.0,
    communication_p=0.15,
    source_equalize_p=0.25,
)
RUN_PRESENCE_TRAINING = True
PRESENCE_DIR = RUN_ROOT / "v2_presence_cnn"
PRESENCE_DIR.mkdir(parents=True, exist_ok=True)


def fit_presence():
    train_ds, train_loader, val_loader, stress_loader = make_task_loaders(
        PRESENCE_CFG["batch"], PRESENCE_CFG["eval_batch"],
        PRESENCE_CFG["rawboost_p"], PRESENCE_CFG["communication_p"], PRESENCE_CFG["source_equalize_p"],
    )
    model = PresenceCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRESENCE_CFG["lr"], weight_decay=PRESENCE_CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRESENCE_CFG["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    best = -float("inf")
    stale = 0
    history = []
    for epoch in range(1, PRESENCE_CFG["epochs"] + 1):
        train_ds.set_epoch(epoch)
        loss = train_epoch(model, train_loader, "presence", optimizer, scaler)
        scheduler.step()
        t_val, p_val = predict_task(model, val_loader, "presence")
        t_stress, p_stress = predict_task(model, stress_loader, "presence")
        normal = presence_report(t_val, p_val)
        stress = presence_report(t_stress, p_stress)
        robust_cps = 0.75 * normal["cps"] + 0.25 * stress["cps"]
        row = {
            "epoch": epoch, "train_loss": loss,
            "normal_cps": normal["cps"], "stress_cps": stress["cps"], "robust_cps": robust_cps,
            "voice_auc": normal["voice_presence_auc"], "music_auc": normal["music_presence_auc"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(PRESENCE_DIR / "history.csv", index=False)
        print(row)

        if robust_cps > best:
            best = robust_cps
            stale = 0
            torch.save({"model_state": model.state_dict(), "config": PRESENCE_CFG, "report": row}, PRESENCE_DIR / "best.pt")
        else:
            stale += 1
        if stale >= PRESENCE_CFG["patience"]:
            break

    del model
    gc.collect(); torch.cuda.empty_cache()


if RUN_PRESENCE_TRAINING:
    fit_presence()



## 15. AASIST Fake3 학습

RawBoost는 기존 0.45에서 **0.15**로 낮추고 communication 0.20, class-independent source equalization 0.35를 사용합니다.


In [ ]:

AASIST_FAKE_CFG = dict(
    batch=16,
    eval_batch=32,
    grad_accum=1,
    epochs=20,
    lr=1e-4,
    weight_decay=1e-4,
    patience=6,
    rawboost_p=0.15,
    communication_p=0.20,
    source_equalize_p=0.35,
)
RUN_AASIST_FAKE_TRAINING = True
AASIST_FAKE_DIR = RUN_ROOT / "v2_aasist_fake3"
AASIST_FAKE_DIR.mkdir(parents=True, exist_ok=True)


def fit_aasist_fake():
    train_ds, train_loader, val_loader, stress_loader = make_task_loaders(
        AASIST_FAKE_CFG["batch"], AASIST_FAKE_CFG["eval_batch"],
        AASIST_FAKE_CFG["rawboost_p"], AASIST_FAKE_CFG["communication_p"], AASIST_FAKE_CFG["source_equalize_p"],
    )
    model = build_aasist_fake3().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AASIST_FAKE_CFG["lr"], weight_decay=AASIST_FAKE_CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=AASIST_FAKE_CFG["epochs"], eta_min=5e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    best = -float("inf")
    stale = 0
    history = []
    for epoch in range(1, AASIST_FAKE_CFG["epochs"] + 1):
        train_ds.set_epoch(epoch)
        loss = train_epoch(model, train_loader, "fake", optimizer, scaler, AASIST_FAKE_CFG["grad_accum"])
        scheduler.step()
        t_val, p_val = predict_task(model, val_loader, "fake")
        t_stress, p_stress = predict_task(model, stress_loader, "fake")
        normal = fake_report(t_val, p_val)
        stress = fake_report(t_stress, p_stress)
        robust_ads = 0.65 * normal["ads"] + 0.35 * stress["ads"]
        row = {
            "epoch": epoch, "train_loss": loss,
            **{f"normal_{k}": v for k, v in normal.items()},
            **{f"stress_{k}": v for k, v in stress.items()},
            "robust_ads": robust_ads,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(AASIST_FAKE_DIR / "history.csv", index=False)
        print(row)

        if robust_ads > best:
            best = robust_ads
            stale = 0
            torch.save({"model_state": model.state_dict(), "config": AASIST_FAKE_CFG, "report": row}, AASIST_FAKE_DIR / "best.pt")
        else:
            stale += 1
        if stale >= AASIST_FAKE_CFG["patience"]:
            break

    del model
    gc.collect(); torch.cuda.empty_cache()


if RUN_AASIST_FAKE_TRAINING:
    fit_aasist_fake()



## 16. XLS-R Graph Fake3 학습

T4 기준 batch 2 + gradient accumulation 8을 기본으로 합니다.  
처음 2 epoch는 XLS-R를 freeze하고 이후 마지막 4 transformer layer만 fine-tune합니다.


In [ ]:

XLSR_FAKE_CFG = dict(
    batch=2,
    eval_batch=4,
    grad_accum=8,
    epochs=10,
    head_lr=8e-5,
    backbone_lr=3e-7,
    weight_decay=1e-4,
    patience=4,
    freeze_epochs=2,
    unfreeze_last=4,
    rawboost_p=0.10,
    communication_p=0.20,
    source_equalize_p=0.35,
)
RUN_XLSR_FAKE_TRAINING = True
XLSR_FAKE_DIR = RUN_ROOT / "v2_xlsr_graph_fake3"
XLSR_FAKE_DIR.mkdir(parents=True, exist_ok=True)


def fit_xlsr_fake():
    from transformers import get_cosine_schedule_with_warmup

    train_ds, train_loader, val_loader, stress_loader = make_task_loaders(
        XLSR_FAKE_CFG["batch"], XLSR_FAKE_CFG["eval_batch"],
        XLSR_FAKE_CFG["rawboost_p"], XLSR_FAKE_CFG["communication_p"], XLSR_FAKE_CFG["source_equalize_p"],
    )
    model = XLSRGraphFake(XLSR_MODEL, freeze=True).to(DEVICE)

    backbone_params = list(model.ssl.parameters())
    backbone_ids = {id(p) for p in backbone_params}
    head_params = [p for p in model.parameters() if id(p) not in backbone_ids]
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": XLSR_FAKE_CFG["backbone_lr"]},
        {"params": head_params, "lr": XLSR_FAKE_CFG["head_lr"]},
    ], weight_decay=XLSR_FAKE_CFG["weight_decay"])

    updates_per_epoch = math.ceil(len(train_loader) / XLSR_FAKE_CFG["grad_accum"])
    total_updates = updates_per_epoch * XLSR_FAKE_CFG["epochs"]
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(10, int(total_updates * 0.08)),
        num_training_steps=max(1, total_updates),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    best = -float("inf")
    stale = 0
    history = []
    for epoch in range(1, XLSR_FAKE_CFG["epochs"] + 1):
        train_ds.set_epoch(epoch)
        if epoch == XLSR_FAKE_CFG["freeze_epochs"] + 1:
            model.unfreeze_last(XLSR_FAKE_CFG["unfreeze_last"])
            print("Unfroze XLS-R last", XLSR_FAKE_CFG["unfreeze_last"], "layers")

        loss = train_epoch(
            model, train_loader, "fake", optimizer, scaler,
            grad_accum=XLSR_FAKE_CFG["grad_accum"], scheduler=scheduler,
        )
        t_val, p_val = predict_task(model, val_loader, "fake")
        t_stress, p_stress = predict_task(model, stress_loader, "fake")
        normal = fake_report(t_val, p_val)
        stress = fake_report(t_stress, p_stress)
        robust_ads = 0.65 * normal["ads"] + 0.35 * stress["ads"]
        row = {
            "epoch": epoch, "train_loss": loss,
            **{f"normal_{k}": v for k, v in normal.items()},
            **{f"stress_{k}": v for k, v in stress.items()},
            "robust_ads": robust_ads,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(XLSR_FAKE_DIR / "history.csv", index=False)
        print(row)

        if robust_ads > best:
            best = robust_ads
            stale = 0
            torch.save({
                "model_state": model.state_dict(),
                "config": XLSR_FAKE_CFG,
                "report": row,
                "xlsr_config": model.ssl.config.to_dict(),
            }, XLSR_FAKE_DIR / "best.pt")
        else:
            stale += 1
        if stale >= XLSR_FAKE_CFG["patience"]:
            break

    del model
    gc.collect(); torch.cuda.empty_cache()


if RUN_XLSR_FAKE_TRAINING:
    fit_xlsr_fake()



## 17. AASIST/XLS-R fake ensemble weight 탐색 + 전체 local score

가중치는 normal validation과 stress validation의 ADS를 함께 사용해 선택합니다.


In [ ]:

ENSEMBLE_CONFIG_PATH = RUN_ROOT / "v2_ensemble.json"


def load_best_presence():
    state = torch.load(PRESENCE_DIR / "best.pt", map_location="cpu", weights_only=False)
    model = PresenceCNN().to(DEVICE)
    model.load_state_dict(state["model_state"], strict=True)
    return model.eval()


def load_best_aasist_fake():
    state = torch.load(AASIST_FAKE_DIR / "best.pt", map_location="cpu", weights_only=False)
    model = build_aasist_fake3().to(DEVICE)
    model.load_state_dict(state["model_state"], strict=True)
    return model.eval()


def load_best_xlsr_fake():
    state = torch.load(XLSR_FAKE_DIR / "best.pt", map_location="cpu", weights_only=False)
    model = XLSRGraphFake(XLSR_MODEL, freeze=True).to(DEVICE)
    model.load_state_dict(state["model_state"], strict=True)
    return model.eval()


# Common validation loaders; no training augment.
_, _, val_loader_common, stress_loader_common = make_task_loaders(
    batch=2, eval_batch=4, rawboost_p=0.0, communication_p=0.0, source_equalize_p=0.0
)

presence_model = load_best_presence()
aasist_model = load_best_aasist_fake()
xlsr_model = load_best_xlsr_fake()

val_targets, val_presence = predict_task(presence_model, val_loader_common, "presence")
_, val_aasist = predict_task(aasist_model, val_loader_common, "fake")
_, val_xlsr = predict_task(xlsr_model, val_loader_common, "fake")

stress_targets, stress_presence = predict_task(presence_model, stress_loader_common, "presence")
_, stress_aasist = predict_task(aasist_model, stress_loader_common, "fake")
_, stress_xlsr = predict_task(xlsr_model, stress_loader_common, "fake")

best = None
for xlsr_weight in np.linspace(0.0, 1.0, 21):
    val_fake = xlsr_weight * val_xlsr + (1 - xlsr_weight) * val_aasist
    stress_fake = xlsr_weight * stress_xlsr + (1 - xlsr_weight) * stress_aasist
    val_ads = fake_report(val_targets, val_fake)["ads"]
    stress_ads = fake_report(stress_targets, stress_fake)["ads"]
    robust_ads = 0.65 * val_ads + 0.35 * stress_ads
    candidate = (robust_ads, xlsr_weight, val_ads, stress_ads)
    if best is None or candidate[0] > best[0]:
        best = candidate

robust_ads, xlsr_weight, val_ads, stress_ads = best
ensemble_cfg = {
    "xlsr_weight": float(xlsr_weight),
    "aasist_weight": float(1 - xlsr_weight),
    "normal_ads": float(val_ads),
    "stress_ads": float(stress_ads),
    "robust_ads": float(robust_ads),
}
ENSEMBLE_CONFIG_PATH.write_text(json.dumps(ensemble_cfg, indent=2), encoding="utf-8")
print("Ensemble:", ensemble_cfg)

val_fake_ensemble = xlsr_weight * val_xlsr + (1 - xlsr_weight) * val_aasist
stress_fake_ensemble = xlsr_weight * stress_xlsr + (1 - xlsr_weight) * stress_aasist

print("Normal full score:", full_report(val_targets, val_fake_ensemble, val_presence))
print("Stress full score:", full_report(stress_targets, stress_fake_ensemble, stress_presence))

del presence_model, aasist_model, xlsr_model
gc.collect(); torch.cuda.empty_cache()



## 18. Audit split 최종 확인 (선택)

Audit는 설정 변경에 사용하지 않는 것이 원칙입니다.  
아래 셀은 최종 선택 후 1회 확인용입니다.


In [ ]:

RUN_AUDIT = True

if RUN_AUDIT:
    audit_dataset = DynamicMixDataset(
        recipe_by_split["audit"], source_by_split["audit"], training=False
    )
    audit_loader = DataLoader(
        audit_dataset, batch_size=4, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda", persistent_workers=False,
    )

    presence_model = load_best_presence()
    aasist_model = load_best_aasist_fake()
    xlsr_model = load_best_xlsr_fake()
    audit_targets, audit_presence = predict_task(presence_model, audit_loader, "presence")
    _, audit_aasist = predict_task(aasist_model, audit_loader, "fake")
    _, audit_xlsr = predict_task(xlsr_model, audit_loader, "fake")
    cfg = json.loads(ENSEMBLE_CONFIG_PATH.read_text(encoding="utf-8"))
    audit_fake = cfg["xlsr_weight"] * audit_xlsr + cfg["aasist_weight"] * audit_aasist
    print("AUDIT:", full_report(audit_targets, audit_fake, audit_presence))
    del presence_model, aasist_model, xlsr_model
    gc.collect(); torch.cuda.empty_cache()



## 19. DACON submit.zip 생성 — Presence-Gated Top-K

### 기존 제출과 다른 점

- `max` 비중 0.6/0.7 pooling을 제거
- 파일당 최대 8개의 4.04초 segment를 전체 구간에서 uniform sampling
- Presence = Top 30% mean
- Voice Fake = Voice Presence가 높은 segment 절반에서 Fake Top 40% weighted mean
- Music Fake = Music Presence가 높은 segment 절반에서 Fake Top 40% weighted mean
- File Fake = Direct File Top-K + presence-gated soft-OR
- AASIST와 XLS-R fake score는 validation에서 찾은 고정 weight로 ensemble

평가 파일끼리의 통계는 사용하지 않고 각 파일 내부 segment만 사용합니다.


In [ ]:

BUILD_SUBMIT_ZIP = True
SUBMIT_ZIP_PATH = DRIVE_ROOT / "submit_v2_domain_robust.zip"

if BUILD_SUBMIT_ZIP:
    required_ckpts = [
        PRESENCE_DIR / "best.pt",
        AASIST_FAKE_DIR / "best.pt",
        XLSR_FAKE_DIR / "best.pt",
        ENSEMBLE_CONFIG_PATH,
    ]
    for path in required_ckpts:
        if not path.exists():
            raise FileNotFoundError(f"먼저 학습/ensemble 셀을 완료하세요: {path}")

    stage = Path("/content/dacon_v2_submit")
    if stage.exists():
        shutil.rmtree(stage)
    model_dir = stage / "model"
    model_dir.mkdir(parents=True)

    presence_ckpt = torch.load(PRESENCE_DIR / "best.pt", map_location="cpu", weights_only=False)
    aasist_ckpt = torch.load(AASIST_FAKE_DIR / "best.pt", map_location="cpu", weights_only=False)
    xlsr_ckpt = torch.load(XLSR_FAKE_DIR / "best.pt", map_location="cpu", weights_only=False)

    torch.save(presence_ckpt["model_state"], model_dir / "presence_weights.pt")
    torch.save(aasist_ckpt["model_state"], model_dir / "aasist_fake_weights.pt")
    torch.save(xlsr_ckpt["model_state"], model_dir / "xlsr_fake_weights.pt")

    # Save HF config only; XLS-R weights are already inside xlsr_fake_weights.pt.
    xlsr_config_dict = xlsr_ckpt["xlsr_config"]
    (model_dir / "xlsr_config.json").write_text(json.dumps(xlsr_config_dict, indent=2), encoding="utf-8")
    shutil.copy2(ENSEMBLE_CONFIG_PATH, model_dir / "ensemble.json")

    # Bundle AASIST source; inference must not access internet.
    shutil.copytree(
        REPO_ROOT / "aasist", model_dir / "aasist",
        ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", "LA", "database"),
    )

    (stage / "script.py").write_text('from __future__ import annotations\n\nimport json\nimport math\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nimport librosa\nimport numpy as np\nimport pandas as pd\nimport soundfile as sf\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchaudio\nfrom transformers import Wav2Vec2Config, Wav2Vec2Model\n\nROOT = Path(__file__).resolve().parent\nMODEL_DIR = ROOT / "model"\nDATA_DIR = ROOT / "data" if (ROOT / "data").exists() else ROOT / "open"\nTEST_DIR = DATA_DIR / "test"\nOUTPUT_DIR = ROOT / "output"\nOUTPUT_PATH = OUTPUT_DIR / "submission.csv"\nSAMPLE_RATE = 16_000\nCLIP_SAMPLES = 64_600\nMAX_SEGMENTS = 8\nAUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".amr"}\nPROBABILITY_COLUMNS = [\n    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",\n    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",\n]\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\nclass PresenceCNN(nn.Module):\n    def __init__(self, dropout=0.20):\n        super().__init__()\n        self.mel = torchaudio.transforms.MelSpectrogram(\n            sample_rate=SAMPLE_RATE, n_fft=1024, win_length=400,\n            hop_length=160, n_mels=96, f_min=20, f_max=7600,\n        )\n        self.encoder = nn.Sequential(\n            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),\n            nn.AdaptiveAvgPool2d(1), nn.Flatten(),\n        )\n        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 2))\n\n    def forward(self, audio):\n        feature = torch.log(self.mel(audio).clamp_min(1e-6))\n        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (feature.std((-2, -1), keepdim=True) + 1e-5)\n        return self.head(self.encoder(feature.unsqueeze(1)))\n\n\nclass AttentionBlock(nn.Module):\n    def __init__(self, dim, dropout=0.1):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(dim)\n        self.attention = nn.MultiheadAttention(dim, 4, dropout=dropout, batch_first=True)\n        self.norm2 = nn.LayerNorm(dim)\n        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))\n\n    def forward(self, x):\n        z = self.norm1(x)\n        x = x + self.attention(z, z, z, need_weights=False)[0]\n        return x + self.ff(self.norm2(x))\n\n\nclass XLSRGraphFake(nn.Module):\n    def __init__(self, config, dim=128, dropout=0.20):\n        super().__init__()\n        self.ssl = Wav2Vec2Model(config)\n        hidden = config.hidden_size\n        self.projection = nn.Linear(hidden, dim)\n        self.feature_projection = nn.Linear(8, dim)\n        self.time_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))\n        self.feature_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))\n        self.head = nn.Sequential(\n            nn.LayerNorm(dim * 4), nn.Dropout(dropout),\n            nn.Linear(dim * 4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 3),\n        )\n\n    def forward(self, audio):\n        audio = (audio - audio.mean(1, keepdim=True)) / (audio.std(1, keepdim=True) + 1e-5)\n        hidden = self.projection(self.ssl(audio).last_hidden_state)\n        time_nodes = F.adaptive_avg_pool1d(hidden.transpose(1, 2), 64).transpose(1, 2)\n        feature_nodes = self.feature_projection(F.adaptive_avg_pool1d(hidden.transpose(1, 2), 8))\n        time_nodes = self.time_graph(time_nodes)\n        feature_nodes = self.feature_graph(feature_nodes)\n        pooled = torch.cat([time_nodes.mean(1), time_nodes.amax(1), feature_nodes.mean(1), feature_nodes.amax(1)], dim=-1)\n        return self.head(pooled)\n\n\ndef build_aasist_fake3():\n    repo = MODEL_DIR / "aasist"\n    if str(repo) not in sys.path:\n        sys.path.insert(0, str(repo))\n    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as f:\n        model_config = json.load(f)["model_config"]\n    from models.AASIST import Model as AASISTModel\n\n    class Wrapper(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.net = AASISTModel(model_config)\n            self.net.out_layer = nn.Linear(self.net.out_layer.in_features, 3)\n\n        def forward(self, audio):\n            _, logits = self.net(audio, Freq_aug=False)\n            return logits\n\n    return Wrapper()\n\n\ndef load_models():\n    presence = PresenceCNN().to(DEVICE)\n    presence.load_state_dict(torch.load(MODEL_DIR / "presence_weights.pt", map_location=DEVICE, weights_only=True))\n    presence.eval()\n\n    aasist = build_aasist_fake3().to(DEVICE)\n    aasist.load_state_dict(torch.load(MODEL_DIR / "aasist_fake_weights.pt", map_location=DEVICE, weights_only=True))\n    aasist.eval()\n\n    config = Wav2Vec2Config.from_json_file(MODEL_DIR / "xlsr_config.json")\n    xlsr = XLSRGraphFake(config).to(DEVICE)\n    xlsr.load_state_dict(torch.load(MODEL_DIR / "xlsr_fake_weights.pt", map_location=DEVICE, weights_only=True))\n    xlsr.eval()\n\n    ensemble = json.loads((MODEL_DIR / "ensemble.json").read_text(encoding="utf-8"))\n    return presence, aasist, xlsr, ensemble\n\n\ndef read_audio(path):\n    try:\n        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)\n        waveform = torch.from_numpy(audio.mean(axis=1))\n    except Exception:\n        try:\n            audio, sample_rate = librosa.load(path, sr=None, mono=True)\n            waveform = torch.from_numpy(np.asarray(audio, dtype=np.float32))\n        except Exception:\n            decoded = subprocess.run(\n                ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(path),\n                 "-ac", "1", "-ar", str(SAMPLE_RATE), "-f", "f32le", "pipe:1"],\n                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=120,\n            )\n            waveform = torch.from_numpy(np.frombuffer(decoded.stdout, dtype="<f4").copy())\n            sample_rate = SAMPLE_RATE\n    if waveform.numel() == 0:\n        raise ValueError(f"empty audio: {path}")\n    if sample_rate != SAMPLE_RATE:\n        waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)\n    return waveform.float().nan_to_num().clamp(-1, 1)\n\n\ndef make_segments(waveform):\n    if waveform.numel() < CLIP_SAMPLES:\n        waveform = waveform.repeat(math.ceil(CLIP_SAMPLES / waveform.numel()))\n        return waveform[:CLIP_SAMPLES].unsqueeze(0)\n    maximum_start = waveform.numel() - CLIP_SAMPLES\n    if maximum_start == 0:\n        return waveform[:CLIP_SAMPLES].unsqueeze(0)\n    count = min(MAX_SEGMENTS, max(1, math.ceil(waveform.numel() / CLIP_SAMPLES)))\n    starts = np.linspace(0, maximum_start, count).round().astype(int).tolist()\n    return torch.stack([waveform[s:s + CLIP_SAMPLES] for s in starts])\n\n\ndef topk_mean(values, fraction=0.30):\n    values = np.asarray(values, dtype=np.float64)\n    k = max(1, int(math.ceil(len(values) * fraction)))\n    idx = np.argsort(values)[-k:]\n    return float(values[idx].mean())\n\n\ndef presence_gated_topk(fake_scores, presence_scores, gate_fraction=0.50, top_fraction=0.40):\n    fake_scores = np.asarray(fake_scores, dtype=np.float64)\n    presence_scores = np.asarray(presence_scores, dtype=np.float64)\n    gate_k = max(1, int(math.ceil(len(fake_scores) * gate_fraction)))\n    gate_idx = np.argsort(presence_scores)[-gate_k:]\n    local_fake = fake_scores[gate_idx]\n    local_presence = presence_scores[gate_idx]\n    top_k = max(1, int(math.ceil(len(local_fake) * top_fraction)))\n    chosen = np.argsort(local_fake)[-top_k:]\n    weights = np.clip(local_presence[chosen], 0.05, 1.0)\n    return float(np.average(local_fake[chosen], weights=weights))\n\n\n@torch.inference_mode()\ndef predict_file(presence_model, aasist_model, xlsr_model, ensemble, waveform):\n    segments = make_segments(waveform)\n\n    presence_outputs = []\n    aasist_outputs = []\n    xlsr_outputs = []\n\n    for start in range(0, len(segments), 16):\n        batch = segments[start:start + 16].to(DEVICE)\n        presence_outputs.append(torch.sigmoid(presence_model(batch)).float().cpu().numpy())\n        aasist_outputs.append(torch.sigmoid(aasist_model(batch)).float().cpu().numpy())\n\n    for start in range(0, len(segments), 8):\n        batch = segments[start:start + 8].to(DEVICE)\n        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):\n            logits = xlsr_model(batch)\n        xlsr_outputs.append(torch.sigmoid(logits.float()).cpu().numpy())\n\n    presence = np.concatenate(presence_outputs, axis=0)\n    aasist_fake = np.concatenate(aasist_outputs, axis=0)\n    xlsr_fake = np.concatenate(xlsr_outputs, axis=0)\n\n    w = float(ensemble["xlsr_weight"])\n    fake = w * xlsr_fake + (1.0 - w) * aasist_fake\n\n    voice_present = topk_mean(presence[:, 0], 0.30)\n    music_present = topk_mean(presence[:, 1], 0.30)\n    voice_fake = presence_gated_topk(fake[:, 1], presence[:, 0], 0.50, 0.40)\n    music_fake = presence_gated_topk(fake[:, 2], presence[:, 1], 0.50, 0.40)\n    direct_file = topk_mean(fake[:, 0], 0.30)\n    coherent_file = 1.0 - (1.0 - voice_fake * voice_present) * (1.0 - music_fake * music_present)\n    file_fake = float(np.clip(0.70 * direct_file + 0.30 * coherent_file, 0.0, 1.0))\n\n    return [\n        file_fake,\n        float(np.clip(voice_fake, 0, 1)),\n        float(np.clip(music_fake, 0, 1)),\n        float(np.clip(voice_present, 0, 1)),\n        float(np.clip(music_present, 0, 1)),\n    ]\n\n\ndef resolve_paths(sample):\n    files = sorted(p for p in TEST_DIR.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES)\n    lookup = {}\n    for path in files:\n        lookup[path.stem] = path\n        lookup[path.name] = path\n    resolved = [lookup.get(str(identifier)) for identifier in sample["ID"]]\n    if any(path is None for path in resolved):\n        if len(files) != len(sample):\n            missing = [str(identifier) for identifier, path in zip(sample["ID"], resolved) if path is None]\n            raise FileNotFoundError(missing[:10])\n        resolved = files\n    return resolved\n\n\ndef main():\n    sample = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8")\n    required = ["ID", *PROBABILITY_COLUMNS]\n    if list(sample.columns) != required:\n        raise ValueError(f"unexpected columns: {list(sample.columns)}")\n\n    presence, aasist, xlsr, ensemble = load_models()\n    rows = []\n    for index, (identifier, path) in enumerate(zip(sample["ID"].astype(str), resolve_paths(sample)), 1):\n        probabilities = predict_file(presence, aasist, xlsr, ensemble, read_audio(path))\n        rows.append([identifier, *probabilities])\n        if index % 50 == 0:\n            print(f"[{index}/{len(sample)}]")\n\n    submission = pd.DataFrame(rows, columns=required)\n    values = submission[PROBABILITY_COLUMNS].to_numpy(dtype=float)\n    if len(submission) != len(sample):\n        raise RuntimeError("row count mismatch")\n    if not np.isfinite(values).all() or not ((values >= 0).all() and (values <= 1).all()):\n        raise ValueError("probability outside [0, 1]")\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    submission.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")\n    print(f"saved {OUTPUT_PATH}: {len(submission)} rows")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
    (stage / "requirements.txt").write_text(
        "# No additional pip install required.\n"
        "# Uses DACON preinstalled: torch, torchaudio, transformers, pandas, numpy, librosa, soundfile.\n",
        encoding="utf-8",
    )

    import py_compile
    py_compile.compile(str(stage / "script.py"), doraise=True)

    if SUBMIT_ZIP_PATH.exists():
        SUBMIT_ZIP_PATH.unlink()
    with zipfile.ZipFile(SUBMIT_ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
        archive.write(model_dir, "model/")
        for path in sorted(model_dir.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(stage).as_posix())
        archive.write(stage / "script.py", "script.py")
        archive.write(stage / "requirements.txt", "requirements.txt")

    with zipfile.ZipFile(SUBMIT_ZIP_PATH) as archive:
        names = archive.namelist()
        top_level = {name.split("/", 1)[0] for name in names}
        assert top_level == {"model", "script.py", "requirements.txt"}
        compressed_gb = SUBMIT_ZIP_PATH.stat().st_size / 1024**3
        uncompressed_gb = sum(info.file_size for info in archive.infolist()) / 1024**3

    if compressed_gb >= 10 or uncompressed_gb >= 32:
        raise RuntimeError({"compressed_gb": compressed_gb, "uncompressed_gb": uncompressed_gb})

    print("created:", SUBMIT_ZIP_PATH)
    print({"compressed_gb": compressed_gb, "uncompressed_gb": uncompressed_gb})



## 20. 제출 스모크 테스트

`RUN_SUBMIT_SMOKE_TEST=True`로 바꾸면 `data/` 구조에서 `script.py`가 실제로 실행되고
`output/submission.csv`가 만들어지는지 확인합니다.

XLS-R가 포함되어 있으므로 첫 로드는 기존 AASIST-only 제출보다 느립니다.


In [ ]:

RUN_SUBMIT_SMOKE_TEST = False
DACON_DUMMY_DATA = Path("/content/dacon_open/data")

if RUN_SUBMIT_SMOKE_TEST:
    smoke_root = Path("/content/dacon_v2_smoke")
    if smoke_root.exists():
        shutil.rmtree(smoke_root)
    smoke_root.mkdir(parents=True)
    with zipfile.ZipFile(SUBMIT_ZIP_PATH) as archive:
        archive.extractall(smoke_root)
    shutil.copytree(DACON_DUMMY_DATA, smoke_root / "data")
    subprocess.run([sys.executable, "script.py"], cwd=smoke_root, check=True, timeout=1800)
    result = pd.read_csv(smoke_root / "output" / "submission.csv")
    sample = pd.read_csv(smoke_root / "data" / "sample_submission.csv")
    assert list(result.columns) == ["ID", *DACON_PROBABILITY_COLUMNS]
    assert result["ID"].astype(str).tolist() == sample["ID"].astype(str).tolist()
    values = result[DACON_PROBABILITY_COLUMNS].to_numpy(dtype=float)
    assert np.isfinite(values).all() and ((0 <= values) & (values <= 1)).all()
    display(result.head())
    print("submit_v2 smoke test passed")



## 21. v2 실행 순서 체크리스트

- [ ] 네 source pool 다운로드/선택 완료
- [ ] FMA vocal screening 완료
- [ ] source manifest / recipe manifest 생성
- [ ] Presence CNN 학습 완료
- [ ] AASIST Fake3 학습 완료
- [ ] XLS-R Graph Fake3 학습 완료
- [ ] normal + stress validation 확인
- [ ] ensemble weight 탐색 완료
- [ ] audit는 최종 1회만 확인
- [ ] `submit_v2_domain_robust.zip` 생성
- [ ] 가능하면 dummy/open data 스모크 테스트

### 리더보드에서 확인할 것

기존 제출은 CPS가 높고 ADS가 낮았습니다. v2에서는 **ADS 상승이 1순위 목표**입니다.
특히 File/Voice/Music EER이 함께 내려가는지 확인하세요.
